# CSI-Bench FallDetection — official ResNet18, Colab

Pinned source: b1beac7b0fa46e01748901195ecfbe309dfc0180. Official Kaggle v12. Train/validation only; no test evaluation. Source-faithful scheduler mismatch is retained and recorded. Results are stored in /content temporarily; download the result archive before ending the session. No Google Drive mount is used.


In [ ]:
import sys, torch
print(sys.version, torch.__version__)
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30)


In [ ]:
# Public upstream code only; no local checkpoint or private file is uploaded.
import os, sys, subprocess, random, json, math, gc
from pathlib import Path
import numpy as np
import torch
COMMIT='b1beac7b0fa46e01748901195ecfbe309dfc0180'
UPSTREAM=Path('/content/csi_bench_official')
if not UPSTREAM.exists():
    subprocess.run(['git','clone','https://github.com/guozhen-jenn-zhu/CSI-Bench-Real-WiFi-Sensing-Benchmark.git',str(UPSTREAM)],check=True)
subprocess.run(['git','-C',str(UPSTREAM),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(UPSTREAM),'rev-parse','HEAD'],text=True).strip()==COMMIT
subprocess.run([sys.executable,'-m','pip','-q','install','remotezip','h5py','pandas','scipy','scikit-learn','seaborn','tqdm'],check=True)
sys.path.insert(0,str(UPSTREAM))
from model.supervised.models import ResNet18Classifier
assert torch.cuda.is_available()
probe_model=None; probe_x=None; probe_loss=None; probe_opt=None
try:
    torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    probe_model=ResNet18Classifier(win_len=500,feature_size=232,num_classes=2,in_channels=1).cuda().train()
    probe_opt=torch.optim.AdamW(probe_model.parameters(),lr=.001,weight_decay=1e-5)
    probe_x=torch.randn(128,1,500,232,device='cuda')
    probe_y=torch.arange(128,device='cuda')%2
    probe_loss=torch.nn.functional.cross_entropy(probe_model(probe_x),probe_y)
    probe_loss.backward();probe_opt.step();torch.cuda.synchronize()
    print('BATCH_128_PROBE_PASS',{'loss':float(probe_loss.detach()),'peak_GiB':torch.cuda.max_memory_allocated()/2**30},flush=True)
    COLAB_BATCH_SIZE=128
except torch.cuda.OutOfMemoryError:
    COLAB_BATCH_SIZE=None
    print('BATCH_128_PROBE_OOM: formal training has NOT started; do not silently change batch size.',flush=True)
finally:
    del probe_model,probe_x,probe_loss,probe_opt
    gc.collect();torch.cuda.empty_cache()
print('SOURCE_COMMIT',COMMIT)
print('FORMAL_BATCH_SIZE',COLAB_BATCH_SIZE)


In [ ]:
import sys
sys.argv=['download_fall.py','--destination','/content/CSI-Bench-v12','--workers','4']
"""Download only FallDetection from the official Kaggle v12 ZIP using HTTP ranges.

Checks ZIP CRC32 and uncompressed size for every file, including resumed files.
No complete 81 GB dataset download, credential, or archive copy is required.
"""
import argparse
import concurrent.futures
import hashlib
import json
from pathlib import Path
import struct
import time
import zlib

import requests
from remotezip import RemoteZip

URL = 'https://www.kaggle.com/api/v1/datasets/download/guozhenjennzhu/csi-bench?datasetVersionNumber=12'


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--destination', type=Path, required=True, help='CSI-Bench-v12 parent directory')
    parser.add_argument('--workers', type=int, default=4)
    args = parser.parse_args()
    root = args.destination.resolve()
    root.mkdir(parents=True, exist_ok=True)
    with RemoteZip(URL, timeout=90) as archive:
        entries = [dict(name=i.filename, size=i.file_size, compressed=i.compress_size,
                        offset=i.header_offset, method=i.compress_type, crc=i.CRC)
                   for i in archive.infolist() if i.filename.startswith('FallDetection/') and not i.is_dir()]
    if not entries:
        raise ValueError('No FallDetection entries in official ZIP')
    entries.sort(key=lambda i: i['offset'])
    (root / 'official_fall_zip_index.json').write_text(json.dumps(entries, indent=2))
    blocks = []
    for item in entries:
        if not blocks or item['offset'] - blocks[-1][0]['offset'] > 16 * 1024**2:
            blocks.append([])
        blocks[-1].append(item)

    def valid(p, item):
        return p.is_file() and p.stat().st_size == item['size'] and zlib.crc32(p.read_bytes()) == item['crc']

    def download(block):
        if all(valid(root / item['name'], item) for item in block):
            return len(block)
        start = block[0]['offset']
        # ZIP local header length can include a 65535-byte extra field.
        last = block[-1]
        end = last['offset'] + 30 + len(last['name'].encode()) + 65535 + last['compressed'] - 1
        for attempt in range(5):
            try:
                with requests.get(URL, headers={'Range': f'bytes={start}-{end}'}, timeout=120, stream=True) as response:
                    response.raise_for_status()
                    if response.status_code != 206 or not response.headers.get('Content-Range', '').startswith(f'bytes {start}-'):
                        raise ValueError('Server did not honor range; refusing full dataset download')
                    chunk = response.content
                for item in block:
                    pos = item['offset'] - start
                    if chunk[pos:pos+4] != b'PK\x03\x04':
                        raise ValueError('Invalid ZIP local header')
                    name_len, extra_len = struct.unpack_from('<HH', chunk, pos+26)
                    data_start = pos + 30 + name_len + extra_len
                    raw = chunk[data_start:data_start+item['compressed']]
                    if item['method'] == 8:
                        raw = zlib.decompress(raw, -15)
                    elif item['method'] != 0:
                        raise ValueError('Unsupported ZIP compression')
                    if len(raw) != item['size'] or zlib.crc32(raw) != item['crc']:
                        raise ValueError(f"CRC or size mismatch: {item['name']}")
                    p = (root / item['name']).resolve()
                    if not p.is_relative_to(root):
                        raise ValueError('Unsafe ZIP member path')
                    p.parent.mkdir(parents=True, exist_ok=True)
                    tmp = p.with_suffix(p.suffix+'.partial')
                    tmp.write_bytes(raw)
                    tmp.replace(p)
                return len(block)
            except Exception:
                if attempt == 4:
                    raise
                time.sleep(2**attempt)
    count = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = [pool.submit(download, block) for block in blocks]
        for future in concurrent.futures.as_completed(futures):
            count += future.result()
            print(f'CRC-verified {count}/{len(entries)} files', flush=True)
    receipt = {'source_url': URL, 'dataset': 'guozhenjennzhu/csi-bench', 'version': 12,
               'downloaded_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
               'files': len(entries), 'uncompressed_bytes': sum(i['size'] for i in entries),
               'verification': 'ZIP CRC32 and exact byte length for every entry',
               'index_sha256': hashlib.sha256((root/'official_fall_zip_index.json').read_bytes()).hexdigest()}
    (root/'download_receipt.json').write_text(json.dumps(receipt, indent=2))
    print(json.dumps(receipt, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/train_resnet18.py
"""Pinned official baseline/Dataset/TaskTrainer, instrumented train/val only.

No test Dataset is constructed. Scheduler advancement preserves upstream code,
including its batch-count/epoch-count mismatch. This is not the corrected variant.
"""
import argparse
import csv
import hashlib
import importlib.metadata
import json
import math
import random
import sys
import time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, f1_score

HERE=Path('/content/csi_bench_official')
MODULE=Path('/content/fall_detection_colab')
sys.path.insert(0,str(HERE))
from model.supervised.models import ResNet18Classifier, MLPClassifier
from load.supervised.benchmark_dataset import BenchmarkCSIDataset
from engine.supervised.task_trainer import TaskTrainer


def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()

def save(p,x):
    tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(x,indent=2));tmp.replace(p)


def strict_collate(batch):
    if any(v is None for v in batch):raise ValueError('Official loader returned a bad record; refusing silent skip')
    return torch.utils.data.default_collate(batch)


def device_metrics(y,p):
    precision,recall,f1,_=precision_recall_fscore_support(y,p,average='binary',zero_division=0)
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel().tolist()
    return dict(n=len(y),fall=int(sum(y)),nonfall=int(len(y)-sum(y)),precision=float(precision),
                recall=float(recall),fall_f1=float(f1),weighted_f1=float(f1_score(y,p,average='weighted',zero_division=0)),
                tp=tp,fp=fp,fn=fn,tn=tn,false_positive_rate=fp/(fp+tn) if fp+tn else 0.)


class ObservedTrainer(TaskTrainer):
    """Only observation and strict coverage checks; super() performs upstream math."""
    def evaluate(self,loader):
        captured=[]
        handle=self.model.register_forward_hook(lambda m,i,o:captured.append(o.detach().cpu()))
        try:loss,acc=super().evaluate(loader)
        finally:handle.remove()
        logits=torch.cat(captured)
        if len(logits)!=len(loader.dataset) or not torch.isfinite(logits).all():
            raise ValueError('Incomplete/nonfinite official validation outputs')
        meta=loader.dataset.split_metadata
        y=(meta['label']=='Fall').astype(int).to_numpy()
        fallidx=loader.dataset.label_mapper.label_to_idx['Fall']
        pred=(logits.argmax(1).numpy()==fallidx).astype(int)
        score=logits.softmax(1)[:,fallidx].tolist()
        records=[dict(sample_id=str(row['id']),device=str(row['device']),label=int(label),fall_score=float(s))
                 for (_,row),label,s in zip(meta.iterrows(),y,score)]
        self.observed_records=records
        epoch=self.current_epoch+1
        summary={'epoch':epoch,'val_loss':loss,'accuracy':acc,
                 'learning_rate_used':self.optimizer.param_groups[0]['lr'],
                 'all':device_metrics(y,pred),'by_device':{}}
        for d in ('HP','ESP32'):
            mask=(meta['device']==d).to_numpy();summary['by_device'][d]=device_metrics(y[mask],pred[mask])
        improved=loss<getattr(self,'observed_best',float('inf'))
        if improved:
            self.observed_best=loss;self.observed_best_epoch=epoch
            with (self.report_path/'best_validation_predictions.csv').open('w',newline='') as f:
                w=csv.DictWriter(f,fieldnames=list(records[0]));w.writeheader();w.writerows(records)
            save(self.report_path/'best_validation_metrics.json',summary)
        self.observed_history.append(summary)
        save(self.report_path/'validation_history.json',self.observed_history)
        save(self.report_path/'progress.json',{'completed':False,'last_epoch':epoch,
            'best_epoch':self.observed_best_epoch,'best_val_loss':self.observed_best,
            'wait':epoch-self.observed_best_epoch,'current':summary})
        print('OBSERVED',json.dumps(summary),flush=True)
        return loss,acc


def main():
    ap=argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--data-root',type=Path,default=Path('/content/CSI-Bench-v12'))
    ap.add_argument('--model',choices=['mlp','resnet18'],default='resnet18')
    ap.add_argument('--run-name',default='official_resnet18_seed42_colab_v1')
    ap.add_argument('--preflight-only',action='store_true')
    args=ap.parse_args()
    report=MODULE/'reports'/args.run_name;ckdir=MODULE/'outputs'/args.run_name
    if (report/'progress.json').exists() or (ckdir/'best_model.pt').exists():raise FileExistsError('Choose a fresh run name')
    report.mkdir(parents=True,exist_ok=True);ckdir.mkdir(parents=True,exist_ok=True)
    import subprocess
    commit='b1beac7b0fa46e01748901195ecfbe309dfc0180'
    assert subprocess.check_output(['git','-C',str(HERE),'rev-parse','HEAD'],text=True).strip()==commit
    assert not subprocess.check_output(['git','-C',str(HERE),'diff','--name-only'],text=True).strip()
    source={'commit':commit}
    random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.set_num_threads(2)
    assert torch.cuda.is_available(), 'This run requires a CUDA GPU'
    device=torch.device('cuda:0');torch.cuda.manual_seed_all(42)
    # Upstream seed helper does not force deterministic algorithms; preserve that.
    datasets={s:BenchmarkCSIDataset(str(args.data_root),'FallDetection',split_name=s,file_format='h5',data_key='CSI_amps') for s in ('train_id','val_id')}
    train,val=datasets.values();assert len(train)==4690 and len(val)==1005
    assert train.label_mapper.label_to_idx==val.label_mapper.label_to_idx=={'Fall':0,'Nonfall':1}
    assert not set(train.split_metadata['id'])&set(val.split_metadata['id'])
    # Do not construct or evaluate any test Dataset. CRCs were checked on download.
    for d in datasets.values():
        assert set(d.split_metadata['id'])==d.split_ids
        assert not d.missing_file_paths
    for d in datasets.values():
        for i in range(len(d)):
            item=d[i]
            if item is None or item[0].shape!=(1,500,232) or not torch.isfinite(item[0]).all():raise ValueError('Data preflight failed')
    # Explicitly demonstrate official normalization/padding on one real record/device.
    import h5py
    samples={}
    for d in ('HP','ESP32'):
        i=int(train.split_metadata.index[train.split_metadata['device']==d][0]);x,y=train[i]
        row=train.split_metadata.iloc[i]
        with h5py.File(args.data_root/'FallDetection'/row['file_path'],'r') as f:raw=torch.from_numpy(f['CSI_amps'][()]).float().permute(2,1,0)
        expected=(raw-raw.mean(dim=(1,2),keepdim=True))/raw.std(dim=(1,2),keepdim=True).clamp_min(1e-8)
        torch.testing.assert_close(x[:,:,:raw.shape[-1]],expected,rtol=0,atol=0)
        assert torch.count_nonzero(x[:,:,raw.shape[-1]:])==0
        samples[d]={'sample_id':str(row['id']),'shape':list(x.shape),'official_label':int(y),
                    'valid_mean':float(expected.mean()),'valid_std':float(expected.std()),'padded_nonzero':0}
    model=(MLPClassifier(win_len=500,feature_size=232,num_classes=2) if args.model=='mlp' else ResNet18Classifier(win_len=500,feature_size=232,num_classes=2,in_channels=1)).to(device)
    model.eval()
    with torch.no_grad():out=model(torch.stack([train[0][0],train[1][0]]).to(device))
    assert out.shape==(2,2) and torch.isfinite(out).all()
    trainloader=DataLoader(train,batch_size=128,shuffle=True,num_workers=0,pin_memory=False,collate_fn=strict_collate)
    valloader=DataLoader(val,batch_size=128,shuffle=False,num_workers=0,pin_memory=False,collate_fn=strict_collate)
    config={'source_commit':source['commit'],'model':model.__class__.__name__, 'pretrained_weights':None,'seed':42,
        'batch_size':128,'max_epochs':100,'patience':15,'optimizer':'AdamW','lr':.001,'weight_decay':1e-5,
        'criterion':'unweighted CrossEntropyLoss','label_mapping':train.label_mapper.label_to_idx,
        'train_n':len(train),'val_n':len(val),'test_evaluated':False,'device':str(device),
        'gpu':torch.cuda.get_device_name(0),
        'platform_deviations':['Colab Tesla T4 instead of A10G','num_workers=0 instead of 4 (explicit Colab adaptation)',
            'strict failure on bad data instead of silently filtering None; all train/val files passed'],
        'scheduler_caveat':'upstream batch-count warmup lambda advanced once per epoch; retained unchanged',
        'warmup_steps':len(trainloader)*5,'scheduler_updates_per_epoch':1,'preflight':samples,
        'data_version':12,'data_receipt':json.loads((args.data_root/'download_receipt.json').read_text()),
        'split_sha256':{s:sha(args.data_root/'FallDetection/splits'/f'{s}.json') for s in datasets},
        'known_data_limitation':'one identical train/validation content pair in official v12 split; retained',
        'source_unchanged':True,
        'versions':{n:importlib.metadata.version(n) for n in ('torch','torchvision','numpy','h5py','pandas','scikit-learn')}}
    save(report/'run_config.json',config);print('CONFIG',json.dumps(config),flush=True)
    if args.preflight_only:return
    optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=1e-5)
    num_steps=len(trainloader)*100;warmup_steps=len(trainloader)*5
    def official_schedule(step):
        if step<warmup_steps:return float(step)/float(max(1,warmup_steps))
        progress=float(step-warmup_steps)/float(max(1,num_steps-warmup_steps))
        return .5*(1.+math.cos(math.pi*progress))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=official_schedule)
    trainer=ObservedTrainer(model,trainloader,val_loader=valloader,test_loader=None,
        criterion=torch.nn.CrossEntropyLoss(),optimizer=optimizer,scheduler=scheduler,
        device=device,save_path=str(ckdir),num_classes=2,label_mapper=train.label_mapper,
        config={'epochs':100,'patience':15,'warmup_epochs':5})
    trainer.report_path=report;trainer.observed_history=[]
    start=time.monotonic();trainer.train()
    checkpoint=torch.load(ckdir/'best_model.pt',map_location='cpu',weights_only=True)
    metrics=json.loads((report/'best_validation_metrics.json').read_text())
    assert checkpoint['epoch']+1==metrics['epoch'] and abs(checkpoint['val_loss']-metrics['val_loss'])<1e-8
    config['model_state_dict']={k:v.cpu() for k,v in checkpoint['model_state_dict'].items()}
    config['best_epoch']=metrics['epoch'];config['best_val_loss']=metrics['val_loss']
    config['split_identifiers']={s:sorted(d.split_metadata['id'].tolist()) for s,d in datasets.items()}
    torch.save(config,ckdir/'reproduction_bundle.pt')
    summary={'completed':True,'epochs_run':len(trainer.observed_history),'best_epoch':metrics['epoch'],
        'best_val_loss':metrics['val_loss'],'best_metrics':metrics,'elapsed_seconds':time.monotonic()-start,
        'test_evaluated':False,'prior_local_artifacts_touched':False,'checkpoint_sha256':sha(ckdir/'best_model.pt')}
    save(report/'training_summary.json',summary);save(report/'progress.json',summary)
    history=trainer.observed_history
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout='constrained')
    epochs=[h['epoch'] for h in history]
    axes[0].plot(epochs,trainer.train_losses,label='train');axes[0].plot(epochs,[h['val_loss'] for h in history],label='val');axes[0].set_ylabel('Unweighted CE')
    for d,ax in zip(('HP','ESP32'),axes[1:]):
        for k in ('precision','recall','fall_f1','weighted_f1'):ax.plot(epochs,[h['by_device'][d][k] for h in history],label=k)
        ax.set_title(d+' validation');ax.set_ylim(0,1.05)
    for ax in axes:ax.axvline(metrics['epoch'],ls='--',color='gray');ax.legend(fontsize=8);ax.set_xlabel('Epoch');ax.grid(alpha=.2)
    fig.savefig(report/'training_curves.png',dpi=150);plt.close(fig)
    print('DONE',json.dumps(summary),flush=True)

if __name__=='__main__':main()


In [ ]:
assert COLAB_BATCH_SIZE == 128, "Resolve memory probe before formal training"
import subprocess
from pathlib import Path
log_path=Path('/content/official_resnet18_training.log')
with log_path.open('w') as log:
    process=subprocess.Popen([sys.executable,'-u','/content/train_resnet18.py'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    returncode=process.wait()
assert returncode==0, f"Training exited with {returncode}; inspect the log, do not fabricate results."


In [ ]:
# Run this cell AFTER training succeeds. New Python process; validation only.
import subprocess, sys, textwrap
from pathlib import Path
reload_code=r'''
import sys, csv, json, hashlib
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
sys.path.insert(0,'/content')
from train_resnet18 import ResNet18Classifier, BenchmarkCSIDataset, strict_collate
root=Path('/content/fall_detection_colab')
name='official_resnet18_seed42_colab_v1'
report=root/'reports'/name
checkpoint=root/'outputs'/name/'best_model.pt'
summary=json.loads((report/'training_summary.json').read_text())
assert summary['completed'] and not summary['test_evaluated']
ck=torch.load(checkpoint,map_location='cpu',weights_only=True)
model=ResNet18Classifier(win_len=500,feature_size=232,num_classes=2,in_channels=1).cuda().eval()
model.load_state_dict(ck['model_state_dict'],strict=True)
val=BenchmarkCSIDataset('/content/CSI-Bench-v12','FallDetection',split_name='val_id',file_format='h5',data_key='CSI_amps')
loader=DataLoader(val,batch_size=128,shuffle=False,num_workers=0,collate_fn=strict_collate)
scores=[]
with torch.inference_mode():
    for x,y in loader:
        scores.extend(model(x.cuda()).softmax(1)[:,0].cpu().tolist())
reference=list(csv.DictReader((report/'best_validation_predictions.csv').open()))
assert [r['sample_id'] for r in reference]==val.split_metadata['id'].astype(str).tolist()
a=np.array(scores);b=np.array([float(r['fall_score']) for r in reference])
diff=float(np.max(np.abs(a-b)))
changed=int(np.sum((a>.5)!=(b>.5)))
assert diff<1e-4 and changed==0
result={'new_python_process':True,'validation_n':len(a),'max_score_difference':diff,'changed_predictions':changed,'best_epoch':int(ck['epoch'])+1,'test_evaluated':False,'checkpoint_sha256':hashlib.sha256(checkpoint.read_bytes()).hexdigest()}
(report/'checkpoint_check.json').write_text(json.dumps(result,indent=2))
print('INDEPENDENT_RELOAD_PASS',json.dumps(result),flush=True)
'''
Path('/content/check_colab_checkpoint.py').write_text(textwrap.dedent(reload_code))
subprocess.run([sys.executable,'-u','/content/check_colab_checkpoint.py'],check=True)
import json, zipfile
run_name='official_resnet18_seed42_colab_v1'
root=Path('/content/fall_detection_colab')
print(json.dumps(json.loads((root/'reports'/run_name/'training_summary.json').read_text()),indent=2))
from IPython.display import Image, display
display(Image(filename=str(root/'reports'/run_name/'training_curves.png')))
archive=Path('/content/CSI-Bench_ResNet18_seed42_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for group in ('reports','outputs'):
        for path in (root/group/run_name).rglob('*'):
            if path.is_file():z.write(path,path.relative_to(root))
    for path in (Path('/content/train_resnet18.py'),Path('/content/check_colab_checkpoint.py'),Path('/content/official_resnet18_training.log'),Path('/content/CSI-Bench-v12/download_receipt.json'),Path('/content/CSI-Bench-v12/official_fall_zip_index.json')):
        z.write(path,path.name)
print('RESULT_ARCHIVE',str(archive),'bytes',archive.stat().st_size)
from google.colab import files
files.download(str(archive))
